In [1]:
import pandas as pd
import pyranges as pr
from scipy.stats import ttest_ind
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
%reload_ext memory_profiler

### **Functions def**

In [2]:
def load_whole_gtf(gtf_file):
    columns = ["Chromosome", "Source", "Feature", "Start", "End", "Score", "Strand", "Frame", "Attribute"]
    gtf = pd.read_csv(gtf_file, sep="\t", comment="#", names=columns)
    gtf["Chromosome"] = 'chr' + gtf["Chromosome"].astype(str)
    return gtf[["Chromosome", "Feature","Start", "End", "Strand", "Attribute"]]

In [3]:
def subset_gtf(gtf_df):
    return gtf_df[gtf_df["Feature"]=='gene']

In [4]:
def load_danio_code(PADRE_file):
    columns = ["Chromosome", "Start", "End", "Regulatory_Element", "Score", "Strand", "thickStart", "thickEnd", "itemRgb"]
    PADRE = pd.read_csv(PADRE_file, sep="\t", names=columns, header=None)
    return PADRE

In [5]:
def filter_PADRE(PADRE_df,regulatory_element_tag='EnhA1'):
    return PADRE_df.loc[PADRE_df['Regulatory_Element'].str.contains(r'{0}'.format(regulatory_element_tag), regex=True)==True]

In [6]:
def percentage(part, whole):
  return 100 * float(part)/float(whole)

### **Files Loading**

In [7]:
## Ensembl Genome Annotation for genes positions
gtf_file = "/path_to_Reference/Ensembl/Danio_rerio.GRCz11.113.filtered.gtf" 
#gtf_file_version_GRCz10 = '../Reference/Ensembl/GRCz10.gtf'

## Danio code for enhancers positions
PADRE_file = "/path_to_Reference/DANIO-CODE_annotations/longPec_PADREs.bed"

In [8]:
## Load genome annotation
features_all = load_whole_gtf(gtf_file)
# keep only genes
features_genes = subset_gtf(features_all)

## Load longPec_PADREs.bed and select only Active enhancers
PADRE_df = load_danio_code(PADRE_file)
enhancers_PADRE = filter_PADRE(PADRE_df,'5_EnhA1') #'5_EnhA1' #'6_EnhFlank' #'7_EnhWk1'
enhancers_PADRE

/local/scratch/tmp/ipykernel_3876605/2565973506.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  gtf = pd.read_csv(gtf_file, sep="\t", comment="#", names=columns)


,Chromosome,Start,End,Regulatory_Element,Score,Strand,thickStart,thickEnd,itemRgb
17,chr19,20216548,20217107,5_EnhA1,1000,.,20216548,20217107,"227,26,28"
22,chr5,32591371,32592502,5_EnhA1,1000,.,32591371,32592502,"227,26,28"
51,chr19,15837578,15838096,5_EnhA1,1000,.,15837578,15838096,"227,26,28"
66,chr5,32545972,32546726,5_EnhA1,1000,.,32545972,32546726,"227,26,28"
69,chr20,26960610,26961073,5_EnhA1,1000,.,26960610,26961073,"227,26,28"
...,...,...,...,...,...,...,...,...,...
148874,chr3,30457795,30458223,5_EnhA1,1000,.,30457795,30458223,"227,26,28"
148876,chr23,5518072,5518565,5_EnhA1,1000,.,5518072,5518565,"227,26,28"
148878,chr16,49185775,49186678,5_EnhA1,1000,.,49185775,49186678,"227,26,28"
148879,chr17,37834847,37835447,5_EnhA1,1000,.,37834847,37835447,"227,26,28"


### **Annotation of enhancers**
What is the proportion of enhancers falling inside genes?

We don't have strand info in Danio Code. **Would strand info be important here tho?** Let's remove the strand column:

In [9]:
features_genes=features_genes.drop(['Strand'], axis=1)
enhancers_PADRE=enhancers_PADRE.drop(['Strand'], axis=1)

In [10]:
features_genes_ranges = pr.PyRanges(features_genes)
enhancers_PADRE_ranges = pr.PyRanges(enhancers_PADRE)

In [11]:
## Inclusion
inclusion_enhancer_in_gene = enhancers_PADRE_ranges.join(features_genes_ranges, how='containment', report_overlap=True)
inclusion_gene_in_enhancer = features_genes_ranges.join(enhancers_PADRE_ranges,how='containment',report_overlap=True)

print(len(inclusion_enhancer_in_gene), len(inclusion_gene_in_enhancer))

## Overlap
overlap = enhancers_PADRE_ranges.join(features_genes_ranges, report_overlap=True) 

print(len(overlap))

30090 0
31166


In [12]:
inclusion_enhancer_in_gene.df['End']-inclusion_enhancer_in_gene.df['Start']


0        490
1        837
2        809
3        710
4        428
        ... 
30085    367
30086    565
30087    481
30088    481
30089    301
Length: 30090, dtype: int64

### **Proportions**

In [13]:
nb_genes_ensembl=len(features_genes)
nb_enhancers_longPec_PADRE=len(enhancers_PADRE)

nb_enhancers_inside_gene=len(inclusion_enhancer_in_gene)
nb_enhancers_genes_overlapping=len(overlap)

In [14]:
proportion_enhancers_in_genes=round(percentage(nb_enhancers_inside_gene,nb_enhancers_longPec_PADRE))
print(f"{proportion_enhancers_in_genes}% of enhancers (as annotated in DANIO CODE) are found inside genes (as annotated by Ensembl)")

60% of enhancers (as annotated in DANIO CODE) are found inside genes (as annotated by Ensembl)


In [15]:
proportion_genes_overlap=round(percentage(nb_enhancers_genes_overlapping,nb_genes_ensembl))
proportion_enhancers_overlap=round(percentage(nb_enhancers_genes_overlapping,nb_enhancers_longPec_PADRE))

print(f"{proportion_genes_overlap}% of Ensembl annotated genes are found in an overlap with a Danio Code enhancer \n{proportion_enhancers_overlap}% of Danio Code enhancers are found in an overlap with an Ensembl gene")

123% of Ensembl annotated genes are found in an overlap with a Danio Code enhancer 
62% of Danio Code enhancers are found in an overlap with an Ensembl gene
